# Collision checking and nearest neighbours in C-space: notebook

The room of RO-009 and the robot's 0.25 m disk footprint of RO-037 (`images/cspace.py`); the checks of this Note are in `images/collide.py`. Here we compute every number of the Note.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
from shapely.geometry import Point, Polygon, LineString
import cspace as c
import collide as k

## 1. The three questions (Note, Section 1)

In [2]:
a, b = np.array([2.0, 2.0]), np.array([4.0, 3.6])
print('centre (3.5, 3.2) free:', k.free(c.TEST), ' clearance', round(k.clearance(c.TEST), 3))
for name, p in [('a', a), ('b', b), ('middle', (a + b) / 2)]:
    print(name, p, 'free:', k.free(p))
print('headings 350 and 15: plain', 350 - 15, 'deg; short way', round(np.degrees(k.angle_dist(np.radians(350), np.radians(15))), 3), 'deg')

centre (3.5, 3.2) free: False  clearance -0.05
a [2. 2.] free: True
b [4.  3.6] free: True
middle [3.  2.8] free: True
headings 350 and 15: plain 335 deg; short way 25.0 deg


## 2. Broad phase, narrow phase, sweep and prune (Note, Section 2)

In [3]:
q = c.TEST
boxes = {n: k.aabb(g) for n, g in k.furniture().items()}
boxes['robot'] = k.aabb(k.robot(q))
for n, bx in boxes.items():
    print(f'{n:11s} x {bx[0]:5.2f} to {bx[2]:5.2f}   y {bx[1]:5.2f} to {bx[3]:5.2f}')
n = len(boxes)
print('bodies', n, 'pairs', n * (n - 1) // 2)
print('boxes overlapping the robot box:', [m for m in boxes if m != 'robot' and k.overlap(boxes[m], boxes['robot'])])
print('narrow phase, distance from the centre to the table:', round(k.furniture()['table'].distance(Point(*q)), 3))
order, xpairs, looks = k.sweep_and_prune(boxes)
print('sorted:', order)
print('pairs overlapping along x (and their full box test):', xpairs)
N = 10_000
print('10,000 boxes: pairs', N * (N - 1) // 2, '; n log2 n =', round(N * np.log2(N)))

table       x  3.00 to  5.50   y  1.00 to  3.00
cabinet     x  5.60 to  7.00   y  2.40 to  4.00
pink pole   x  0.90 to  1.10   y  2.90 to  3.10
green pole  x  5.90 to  6.10   y  0.70 to  0.90
robot       x  3.25 to  3.75   y  2.95 to  3.45
bodies 5 pairs 10
boxes overlapping the robot box: ['table']
narrow phase, distance from the centre to the table: 0.2
sorted: ['pink pole', 'table', 'robot', 'cabinet', 'green pole']
pairs overlapping along x (and their full box test): [('table', 'robot', True), ('cabinet', 'green pole', False)]
10,000 boxes: pairs 49995000 ; n log2 n = 132877


## 3. Bounding-volume hierarchy (Note, Section 3)

In [4]:
items = [(nm, k.aabb(g)) for nm, g in k.all_bodies().items()]
root = k.build_bvh(items)
def depth(nd): return 0 if 'name' in nd else 1 + max(depth(kid) for kid in nd['kids'])
def count(nd): return 1 if 'name' in nd else 1 + sum(count(kid) for kid in nd['kids'])
print('bodies (leaves):', len(items), ' wall panels:', len(k.walls()), ' boxes in the tree:', count(root), ' levels below the root:', depth(root))
for q in [c.TEST, c.START[:2]]:
    log = []
    hits = k.query_bvh(root, k.aabb(k.robot(q)), log)
    print(q, 'box tests:', len(log), ' bodies left for the narrow phase:', hits)
rng = np.random.default_rng(0)
tests = []
for _ in range(2000):
    q = (rng.uniform(0.25, 6.75), rng.uniform(0.25, 3.75))
    log = []
    k.query_bvh(root, k.aabb(k.robot(q)), log)
    tests.append(len(log))
print('2,000 random centres: mean box tests', round(np.mean(tests), 1), ' log2(25) =', round(np.log2(25), 2))
print('log2 of 10,000 and 1,000,000:', round(np.log2(1e4), 1), round(np.log2(1e6), 1))

bodies (leaves): 25  wall panels: 21  boxes in the tree: 49  levels below the root: 5
(3.5, 3.2) box tests: 9  bodies left for the narrow phase: ['table']
(2.0, 1.0) box tests: 11  bodies left for the narrow phase: []
2,000 random centres: mean box tests 12.0  log2(25) = 4.64
log2 of 10,000 and 1,000,000: 13.3 19.9


## 4. Self-collision (Note, Section 4)

In [5]:
for th in [(0, 90, 90), (0, 150, 150)]:
    p = k.arm(np.radians(th))
    print(th, 'joints', [tuple(round(v, 3) for v in pt) for pt in p], 'links 1 and 3 cross:', k.self_collides(np.radians(th)))
p = k.arm(np.radians((0, 150, 150)))
x = LineString(p[:2]).intersection(LineString(p[2:]))
print('crossing point:', round(x.x, 3), round(x.y, 3))
rng = np.random.default_rng(0)
TH = rng.uniform(-np.pi, np.pi, (10_000, 3))
hits = sum(k.self_collides(t) for t in TH)
print('10,000 random joint angles: links 1 and 3 cross in', hits, '=', round(100 * hits / 10_000, 2), 'percent')
for m in (3, 7):
    print(m, 'links: pairs', m * (m - 1) // 2, ' joined by a joint', m - 1, ' left to check', m * (m - 1) // 2 - (m - 1))

(0, 90, 90) joints [(0.0, 0.0), (np.float64(1.0), np.float64(0.0)), (np.float64(1.0), np.float64(0.8)), (np.float64(0.4), np.float64(0.8))] links 1 and 3 cross: False
(0, 150, 150) joints [(0.0, 0.0), (np.float64(1.0), np.float64(0.0)), (np.float64(0.307), np.float64(0.4)), (np.float64(0.607), np.float64(-0.12))] links 1 and 3 cross: True
crossing point: 0.538 0.0


10,000 random joint angles: links 1 and 3 cross in 735 = 7.35 percent
3 links: pairs 3  joined by a joint 2  left to check 1
7 links: pairs 21  joined by a joint 6  left to check 15


## 5. Distance queries (Note, Section 5)

In [6]:
s = np.array(c.START[:2])
T = Polygon(c.TABLE)
print('start: distance to the table', round(T.distance(Point(*s)) - c.R, 3), '; to the bottom wall', 1.0 - c.R, '; clearance', round(k.clearance(s), 3))
d = np.array([1.0, 0.0])
print('table corners dotted with -d:', [float(-d @ np.array(p)) for p in c.TABLE])
sup_t = max(c.TABLE, key=lambda p: float(-d @ np.array(p)))
sup_r = s + c.R * d
print('support points: robot', sup_r, 'table', sup_t, ' difference', sup_r - np.array(sup_t))
Dm = Polygon([tuple(s - np.array(p)) for p in c.TABLE]).buffer(c.R, quad_segs=64)
print('Minkowski difference robot - table: contains the origin', Dm.contains(Point(0, 0)), '; distance to the origin', round(Dm.distance(Point(0, 0)), 3))
print('penetration depth at (3.5, 3.2):', round(-k.clearance(c.TEST), 3))

start: distance to the table 0.75 ; to the bottom wall 0.75 ; clearance 0.75
table corners dotted with -d: [-3.0, -5.0, -5.5, -3.5]
support points: robot [2.25 1.  ] table (3.0, 1.0)  difference [-0.75  0.  ]
Minkowski difference robot - table: contains the origin False ; distance to the origin 0.75
penetration depth at (3.5, 3.2): 0.05


## 6. Checking a segment (Note, Section 6)

In [7]:
a, b = np.array([2.0, 2.0]), np.array([4.0, 3.6])
L = np.linalg.norm(b - a)
print('length', round(L, 3))
ts = np.linspace(0, 1, 10_001)
bad = [t for t in ts if not k.free(a + t * (b - a))]
print('in collision from t =', round(bad[0], 3), 'to', round(bad[-1], 3), ':', round((bad[-1] - bad[0]) * L, 3), 'm')
n = int(np.ceil(L / 0.6))
print('epsilon 0.6 m: n =', n, 'steps of', round(L / n, 3), 'm; checks free:', [k.free(a + t * (b - a)) for t in np.arange(n + 1) / n])
print('halvings to get under 0.1 m:', [round(L / 2 ** j, 3) for j in range(6)])
even = [i / 32 for i in range(33)]
half = k.halving_order(5)
print('from a: first hit at check', next(i for i, t in enumerate(even) if not k.free(a + t * (b - a))) + 1)
print('halving order', [round(t, 4) for t in half[:6]], '... first hit at check', next(i for i, t in enumerate(half) if not k.free(a + t * (b - a))) + 1)

length 2.561


in collision from t = 0.62 to 0.777 : 0.403 m
epsilon 0.6 m: n = 5 steps of 0.512 m; checks free: [True, True, True, True, True, True]
halvings to get under 0.1 m: [np.float64(2.561), np.float64(1.281), np.float64(0.64), np.float64(0.32), np.float64(0.16), np.float64(0.08)]
from a: first hit at check 21
halving order [0.0, 1.0, 0.5, 0.25, 0.75, 0.125] ... first hit at check 5


In [8]:
a2, b2 = np.array(c.START[:2]), np.array(c.GOAL)
log = []
print('drive to the door free (clearance method):', k.exact_check(a2, b2, log), ';', len(log), 'checks')
for p, cl in log:
    print('   ', tuple(round(v, 3) for v in p), 'clearance', round(cl, 3))
print('even checks every 0.1 m on 3.4 m:', int(np.ceil(3.4 / 0.1)) + 1)

drive to the door free (clearance method): True ; 9 checks
    (np.float64(2.0), np.float64(1.0)) clearance 0.75
    (np.float64(3.6), np.float64(4.0)) clearance 0.35
    (np.float64(2.8), np.float64(2.5)) clearance 0.308
    (np.float64(2.4), np.float64(1.75)) clearance 0.514
    (np.float64(2.6), np.float64(2.125)) clearance 0.411
    (np.float64(3.2), np.float64(3.25)) clearance 0.141
    (np.float64(3.0), np.float64(2.875)) clearance 0.205
    (np.float64(3.1), np.float64(3.062)) clearance 0.153
    (np.float64(3.4), np.float64(3.625)) clearance 0.298
even checks every 0.1 m on 3.4 m: 35


## 7. Distances with headings (Note, Section 7)

In [9]:
r = np.radians
for t1, t2 in [(350, 15), (0, 360), (350, 0), (0, 15)]:
    print(t1, t2, 'plain', abs(t1 - t2), ' angular distance', round(np.degrees(k.angle_dist(r(t1), r(t2))), 3))
Q = (3.55, r(350))
for name, p in [('E', (3.60, r(15))), ('H', (3.40, r(320)))]:
    print(name, 'heading part', round(k.W_RAD * k.angle_dist(Q[1], p[1]), 4), 'm; distance', round(k.config_dist(Q, p), 3), 'm; plain', round(k.config_dist(Q, p, wrap=False), 3), 'm')

350 15 plain 335  angular distance 25.0
0 360 plain 360  angular distance 0.0
350 0 plain 350  angular distance 10.0
0 15 plain 15  angular distance 15.0
E heading part 0.1091 m; distance 0.12 m; plain 1.463 m
H heading part 0.1309 m; distance 0.199 m; plain 0.199 m


## 8. k-d tree search with wrap-around (Note, Section 8)

In [10]:
P = [('A', 3.10, 40), ('B', 3.30, 120), ('C', 3.55, 200), ('D', 3.85, 300), ('E', 3.60, 15), ('F', 3.20, 250), ('G', 3.90, 150), ('H', 3.40, 320)]
root = k.build_kd([(x, r(t), nm) for nm, x, t in P])
for wrap in (False, True):
    log = []
    best = k.search_kd(root, Q, wrap, (None, np.inf), log)
    print('wrap' if wrap else 'plain', '-> nearest', best[0][2], round(best[1], 3), 'm')
    for cell, kind, bst, p, dd in log:
        print('    ', kind, p[2] if p else '', round(dd, 3), 'cell headings', [round(np.degrees(cell[0][1])), round(np.degrees(cell[1][1]))])
print('heading gap to the cell 0..175 deg: plain', round(k.W_RAD * r(175), 3), 'm; round the circle', round(k.W_RAD * r(10), 3), 'm')

plain -> nearest H 0.199 m
     leaf C 0.654 cell headings [175, 360]
     leaf D 0.371 cell headings [175, 360]
     pruned  0.764 cell headings [0, 175]
     leaf H 0.199 cell headings [185, 360]
     pruned  0.25 cell headings [185, 360]
     pruned  0.724 cell headings [0, 185]
wrap -> nearest E 0.12 m
     leaf C 0.654 cell headings [175, 360]
     leaf D 0.371 cell headings [175, 360]
     leaf E 0.12 cell headings [0, 175]
     pruned  0.205 cell headings [0, 175]
     leaf H 0.199 cell headings [185, 360]
     pruned  0.25 cell headings [185, 360]
     leaf B 0.62 cell headings [0, 185]
     pruned  0.353 cell headings [0, 185]
heading gap to the cell 0..175 deg: plain 0.764 m; round the circle 0.044 m


## 9. Random directions and rotations (Note, Section 9)

In [11]:
S = k.sample_directions()
print('2D, square: within 5 deg of a diagonal', round(k.near_share(S['square'], [45, 135, 225, 315]), 3), ', of an axis', round(k.near_share(S['square'], [0, 90, 180, 270]), 3), ', fair', round(40 / 360, 3))
print('2D, normal: diagonal', round(k.near_share(S['normal2'], [45, 135, 225, 315]), 3), ', axis', round(k.near_share(S['normal2'], [0, 90, 180, 270]), 3))
for nm in ('latitude', 'height', 'normal3', 'rotated'):
    print(nm, 'share with |z| > 0.9:', round(float(np.mean(np.abs(S[nm][:, 2]) > 0.9)), 3))
print('exact for uniform latitude:', round((90 - np.degrees(np.arcsin(0.9))) / 90, 3))
rng = np.random.default_rng(1)
th = rng.uniform(0, 360, 100_000)
print('uniform heading: share in 10..40 deg', round(np.mean((th >= 10) & (th < 40)), 3), '; in 200..230 deg', round(np.mean((th >= 200) & (th < 230)), 3), '; 30/360 =', round(30 / 360, 3))
def Rz(t): return np.array([[np.cos(t), -np.sin(t), 0], [np.sin(t), np.cos(t), 0], [0, 0, 1]])
def Ry(t): return np.array([[np.cos(t), 0, np.sin(t)], [0, 1, 0], [-np.sin(t), 0, np.cos(t)]])
def Rx(t): return np.array([[1, 0, 0], [0, np.cos(t), -np.sin(t)], [0, np.sin(t), np.cos(t)]])
E = np.random.default_rng(2).uniform(0, 2 * np.pi, (20_000, 3))
z = np.array([(Rz(a) @ Ry(b) @ Rx(cc) @ np.array([0, 0, 1.0]))[2] for a, b, cc in E])
print('three uniform angles: share with |z| > 0.9', round(float(np.mean(np.abs(z) > 0.9)), 3))
print('arcsin 0.9 =', round(np.degrees(np.arcsin(0.9)), 1), 'deg')
h = k.shoemake(0.3, 0.6, 0.9)
print('a Shoemake quaternion', np.round(h, 3), 'length', round(float(np.linalg.norm(h)), 6))

2D, square: within 5 deg of a diagonal 0.162 , of an axis 0.086 , fair 0.111
2D, normal: diagonal 0.111 , axis 0.111
latitude share with |z| > 0.9: 0.287
height share with |z| > 0.9: 0.101
normal3 share with |z| > 0.9: 0.101
rotated share with |z| > 0.9: 0.1
exact for uniform latitude: 0.287
uniform heading: share in 10..40 deg 0.084 ; in 200..230 deg 0.083 ; 30/360 = 0.083


three uniform angles: share with |z| > 0.9 0.065
arcsin 0.9 = 64.2 deg
a Shoemake quaternion [-0.492 -0.677 -0.322  0.443] length 1.0
